In [1]:
# Try it out
# Load dataset

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

file_path = "./Titanic-Dataset.csv"

titanic_full: pd.DataFrame = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "yasserh/titanic-dataset",
  file_path,
)

print(titanic_full.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


In [ ]:
# Shuffle and split a dataset
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(
    titanic_full, test_size=0.2, stratify=titanic_full["Pclass"], random_state=42
)
# X train
titanic = train_set.drop("Survived", axis=1)
# y train
titanic_survived_labels = train_set["Survived"].copy()

# X_test
test = test_set.drop("Survived", axis=1)
# y_test 
test_survived_labels = test_set["Survived"].copy()

In [ ]:
# Transformation Pipelines
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

titanic_num = titanic.select_dtypes(include=[np.number])

# для числовых — медиана. SimpleImputer заполнение пропусков (age)
median_num_pipeline = make_pipeline(SimpleImputer(strategy="median"))

# для категориальных — самое частое значение (Embarked)
mf_num_pipeline = make_pipeline(SimpleImputer(strategy="most_frequent"))

# Format text data to numerical data (encoding)
ordinal_enc_pipeline = make_pipeline(OrdinalEncoder())

# логаримф значения
log_pipeline = make_pipeline(FunctionTransformer(np.log, inverse_func=np.exp, feature_names_out="one-to-one"))

preprocessing = ColumnTransformer(
    [
		# Если не будет значения, то заполнить медианой, либо берем ориг
        ("median_num", median_num_pipeline, ["Pclass", "Age", "SibSp", "Parch"]),
		# Для Embarked заполняем тот что чаще встречается
        ("mf_num", mf_num_pipeline, ["Embarked"]),
		# Преобразуем в 0 и 1
		("ord_enc", ordinal_enc_pipeline, ["Sex"]),
		# Логаримф для Fare (цены билета)
		("log", log_pipeline, ["Fare"])
    ]
)
column_prepared = preprocessing.fit_transform(titanic)
print(column_prepared[:2])

# Recover DataFrame
df_prepared = pd.DataFrame(
    column_prepared,
    columns=preprocessing.get_feature_names_out(),
    index=titanic.index,
)
print(df_prepared[:5])


# TODO Добить за Embarked
# Поэтому Embarked всё же оставляем как feature — он несёт информацию
# Warning divide by zero encountered in log — 
# это значит что есть пассажиры с Fare = 0. log(0) = -infinity

[[1.0 52.0 1.0 1.0 'S' 0.0 4.537961436294641]
 [2.0 31.0 0.0 0.0 'S' 1.0 2.3513752571634776]]
    median_num__Pclass median_num__Age median_num__SibSp median_num__Parch  \
820                1.0            52.0               1.0               1.0   
439                2.0            31.0               0.0               0.0   
821                3.0            27.0               0.0               0.0   
403                3.0            28.0               1.0               0.0   
343                2.0            25.0               0.0               0.0   

    mf_num__Embarked ord_enc__Sex log__Fare  
820                S          0.0  4.537961  
439                S          1.0  2.351375  
821                S          1.0  2.159003  
403                S          1.0   2.76317  
343                S          1.0  2.564949  


/home/sieugene/Desktop/work/handson-ml3/venv/lib/python3.10/site-packages/pandas/core/internals/blocks.py:366: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
/home/sieugene/Desktop/work/handson-ml3/venv/lib/python3.10/site-packages/pandas/core/internals/blocks.py:366: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
